<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #4f46e5; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Bases de Conocimiento en Python 🗂️🐍
      </h1>
      <p style="margin: 6px 0 0 0; color: #4f46e5; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Introducción a la Inteligencia Artificial
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #4f46e5; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 05 • Agentes de Conocimiento
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #4f46e5; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>


---
## 🎯 Objetivos de Aprendizaje

En el cuaderno anterior construimos las piezas fundamentales: símbolos, conectivos y `model_check`. Ahora aprenderás la **receta práctica** para implementar bases de conocimiento completas en Python, resolviendo dos mini-casos de aplicación reales con `logic.py`:

1. Un acertijo de **razonamiento por descarte** (`medico_o_clase.py`).
2. Un agente que decide **cómo se dicta una clase según el clima** (`programando.py`).

Ambos casos están disponibles como scripts standalone en la raíz del módulo, y en este cuaderno los importaremos y ejecutaremos para entender paso a paso cómo se construyen.


---
## 1. Implementación de Bases de Conocimiento en Python (1.17.6)

Construir una base de conocimiento con `logic.py` sigue siempre el mismo patrón de 3 pasos:

| Paso | Pregunta que responde | Construcción en `logic.py` |
|---|---|---|
| **1. Modelar el vocabulario** | ¿Qué hechos atómicos son relevantes? | Crear un `Symbol("Nombre")` por cada hecho. |
| **2. Codificar las reglas** | ¿Qué relaciones lógicas conectan esos hechos? | Combinar símbolos con `And`, `Or`, `Not`, `Implication`, `Biconditional`. |
| **3. TELL a la KB** | ¿Cómo agrupo todo el conocimiento? | Envolver todas las sentencias en un único `And(*sentencias)`. |

Una vez armada la KB, el paso de **ASK** siempre es el mismo: `model_check(knowledge, consulta)`.

Veamos este patrón aplicado a dos escenarios completos.


---
## 2. Caso de Aplicación 1: ¿Médica de turno o en clase? 🩺📚

**Escenario:** Camila es estudiante de la Especialización y, además, médica general. Un día cualquiera puede estar de turno en el hospital **o** tener clase de anatomía, pero **nunca ambas cosas a la vez**. Sabemos con certeza que hoy es miércoles, día en que **siempre** tiene clase de anatomía obligatoria.

Este es un clásico ejemplo de **razonamiento por descarte**: a partir de una restricción de exclusión mutua y un hecho observado, el motor de inferencia debe deducir el resto por sí solo, sin que nadie se lo diga explícitamente.

El archivo [`medico_o_clase.py`](medico_o_clase.py) contiene esta base de conocimiento ya resuelta. Vamos a importarlo y reutilizar directamente sus objetos:


In [ ]:
import sys, os
sys.path.append(os.getcwd())

# Importamos el módulo standalone: al importar (a diferencia de ejecutar
# "python3 medico_o_clase.py"), el bloque `if __name__ == "__main__":` NO se
# ejecuta, pero sí quedan disponibles sus símbolos y su base de conocimiento.
import medico_o_clase as caso1

print(f"Base de conocimiento:\n  {caso1.knowledge.formula()}\n")

from logic import Not, model_check

preguntas = {
    "¿Camila NO es médica de turno hoy?": Not(caso1.MD),
    "¿Camila SÍ es médica de turno hoy?": caso1.MD,
    "¿Camila tiene clase de anatomía hoy?": caso1.CL,
}

for pregunta, consulta in preguntas.items():
    print(f"{pregunta:<42} -> {model_check(caso1.knowledge, consulta)}")


Nótese que la clave del razonamiento está en la sentencia `Not(And(MD, CL))` (exclusión mutua): en cuanto sabemos que `CL` (clase de anatomía) es verdadera, la **única** forma de que la KB completa siga siendo consistente es que `MD` (médica de turno) sea falsa. `model_check` descubre esto automáticamente, enumerando los modelos posibles.

---
## 3. Caso de Aplicación 2: El agente del laboratorio de programación 🌧️💻

**Escenario:** un pequeño agente decide cómo se dicta la sesión de laboratorio según el clima, con tres reglas:

1. Hay clase de campo **si y solo si** no llueve (`Biconditional`).
2. Si **no** hay clase de campo, la sesión se dicta de manera **virtual** (`Implication`).
3. El reporte meteorológico de hoy confirma que **sí está lloviendo** (hecho observado).

El archivo [`programando.py`](programando.py) implementa este escenario. Importémoslo y verifiquemos la cadena de inferencia completa:


In [ ]:
import programando as caso2

print(f"Base de conocimiento:\n  {caso2.knowledge.formula()}\n")

preguntas2 = {
    "¿Llueve hoy?": caso2.LLUEVE,
    "¿Hay clase de campo hoy?": caso2.CLASE_CAMPO,
    "¿La sesión se dicta de forma virtual?": caso2.SESION_VIRTUAL,
}

for pregunta, consulta in preguntas2.items():
    print(f"{pregunta:<42} -> {model_check(caso2.knowledge, consulta)}")


```mermaid
flowchart LR
    A["Hecho observado:\nLlueve = True"] --> B["Regla 1 (Biconditional):\nClaseDeCampo ↔ ¬Llueve"]
    B --> C["Se deduce:\nClaseDeCampo = False"]
    C --> D["Regla 2 (Implication):\n¬ClaseDeCampo → SesionVirtual"]
    D --> E["Se deduce:\nSesionVirtual = True"]
```

Este es el valor real de un agente basado en conocimiento: **encadenar** varias reglas independientes para llegar a una conclusión que ningún hecho por sí solo revela directamente.


---
## 4. TELL y ASK: Construyendo un Agente Reutilizable

En el cuaderno anterior definimos la clase `AgenteDeConocimiento`. Vamos a reutilizar la misma idea aquí, pero mostrando algo importante: una KB puede **crecer dinámicamente** a medida que el agente percibe nuevos hechos (`TELL`), y las respuestas a `ASK` se actualizan en consecuencia.


In [ ]:
from logic import Symbol, And, Implication, Not, Or

class AgenteDeConocimiento:
    def __init__(self):
        self._sentencias = []

    def tell(self, sentencia):
        self._sentencias.append(sentencia)

    def base_de_conocimiento(self):
        if not self._sentencias:
            raise ValueError("La base de conocimiento está vacía.")
        return And(*self._sentencias) if len(self._sentencias) > 1 else self._sentencias[0]

    def ask(self, consulta):
        return model_check(self.base_de_conocimiento(), consulta)


# --- Escenario: agente de matrícula académica -----------------------------
PrerequisitoAprobado = Symbol("PrerequisitoAprobado")
CupoDisponible = Symbol("CupoDisponible")
PuedeMatricular = Symbol("PuedeMatricular")

agente = AgenteDeConocimiento()
agente.tell(Implication(And(PrerequisitoAprobado, CupoDisponible), PuedeMatricular))

print("--- Estado inicial (solo la regla, sin hechos observados) ---")
print(f"ASK -> ¿Puede matricular? {agente.ask(PuedeMatricular)}")

print("\n--- El agente percibe: el prerrequisito está aprobado ---")
agente.tell(PrerequisitoAprobado)
print(f"ASK -> ¿Puede matricular? {agente.ask(PuedeMatricular)}")

print("\n--- El agente percibe: además, hay cupo disponible ---")
agente.tell(CupoDisponible)
print(f"ASK -> ¿Puede matricular? {agente.ask(PuedeMatricular)}")


Observa cómo la respuesta a la **misma** consulta (`PuedeMatricular`) cambia a medida que el agente acumula percepciones (`TELL`): pasa de "no se puede inferir" a "se infiere verdadero" en cuanto la KB contiene suficiente evidencia. Así es exactamente como opera el ciclo TELL/ASK en un agente basado en conocimiento real.

---
## Resumen / Puntos Clave

* Toda base de conocimiento en `logic.py` se construye con el mismo patrón: **símbolos → reglas → `And(*sentencias)` → `model_check`**.
* El caso `medico_o_clase.py` ilustra **razonamiento por descarte**: una restricción de exclusión mutua (`Not(And(...))`) combinada con un hecho observado permite deducir el valor de otro símbolo sin que se haya afirmado explícitamente.
* El caso `programando.py` ilustra **encadenamiento de reglas**: un bicondicional y una implicación se combinan para propagar un hecho observado (`Llueve`) hasta una conclusión final (`SesionVirtual`).
* Una KB puede **crecer dinámicamente**: cada llamada a `TELL` añade una sentencia nueva, y las respuestas de `ASK` reflejan siempre el estado *actual* de la base de conocimiento.
* Ambos scripts (`medico_o_clase.py`, `programando.py`) son reutilizables como módulos de Python: se pueden `import`ar para reutilizar sus símbolos y su KB, o ejecutar directamente con `python3 archivo.py` para ver su demostración en consola.

### 🧠 Autoevaluación

1. En `medico_o_clase.py`, ¿qué pasaría con la inferencia si eliminamos la restricción `Not(And(MD, CL))` de la base de conocimiento? Razona tu respuesta antes de probarlo en código.
2. En `programando.py`, ¿por qué se usó un `Biconditional` para la regla 1 y no simplemente una `Implication`? ¿Cambiaría el resultado si fuera una implicación en un solo sentido?
3. Modifica el escenario del agente de matrícula para agregar una tercera condición (por ejemplo, "no tener sanciones disciplinarias") y verifica cómo cambia el resultado de `ask(PuedeMatricular)` a medida que la KB recibe nuevos `TELL`.
4. Diseña un mini-caso propio (3-4 símbolos) inspirado en tu área de trabajo y constrúyelo siguiendo el patrón de 3 pasos explicado en la sección 1.
